# Detection Pass for Local Review

Runs newspaper-navigator at threshold 0.3 on a stratified sample of pages.
I save the raw detections to JSON, which I download and feed into the local
Tkinter reviewer (review_labels.py) to mark FPs and FNs.

Setup:
1. Sync pdfs/ to Drive via Google Drive Desktop
2. Mount Drive, run all cells, download detections.json

### Dependencies

In [1]:
import subprocess, sys
from pathlib import Path

def sh(cmd): subprocess.run(cmd, shell=True, check=True)

sh("apt-get install -qq -y poppler-utils")
sh(f"{sys.executable} -m pip install -q pdf2image opencv-python-headless pillow scikit-learn tqdm")

try:
    import torch
except ImportError:
    sh(f"{sys.executable} -m pip install -q torch torchvision")

import torch
torch_ver = torch.__version__.split("+")[0]
cuda_tag  = ("cu" + torch.version.cuda.replace(".", "")) if torch.cuda.is_available() else "cpu"
print(f"torch {torch_ver} | {cuda_tag}")

try:
    import detectron2
except ImportError:
    wheel_index = (f"https://dl.fbaipublicfiles.com/detectron2/wheels/"
                   f"{cuda_tag}/torch{torch_ver}/index.html")
    result = subprocess.run(
        f"{sys.executable} -m pip install -q detectron2 -f {wheel_index}", shell=True)
    if result.returncode != 0:
        sh(f"{sys.executable} -m pip install -q --no-build-isolation "
           "'git+https://github.com/facebookresearch/detectron2.git'")

weights_path = Path.home() / "newspaper_navigator_model" / "model_final.pth"
weights_path.parent.mkdir(parents=True, exist_ok=True)
if not weights_path.exists():
    sh(f"wget -q -O {weights_path} "
       "https://github.com/LibraryOfCongress/newspaper-navigator/releases/"
       "download/v1.0.0/model_final.pth")
print("weights ready")

torch 2.10.0 | cu128
weights ready


In [2]:
import detectron2
print(detectron2.__version__)

from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.model_zoo import model_zoo
print("detectron2 imports OK")

0.6
detectron2 imports OK


### Mount Drive & Check PDFs

In [21]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=False)

Mounted at /content/drive/


In [26]:
from pathlib import Path

PDF_DIR = Path("/content/drive/Othercomputers/My Laptop/pdfs")
pdf_count = len(list(PDF_DIR.rglob("*.pdf")))
print(f"{pdf_count} PDFs available at {PDF_DIR}")

416 PDFs available at /content/drive/Othercomputers/My Laptop/pdfs


### Config

In [28]:
import torch
from pathlib import Path

PDF_DIR    = Path("/content/drive/Othercomputers/My Laptop/pdfs")
OUTPUT_DIR = Path("/content/drive/MyDrive/detection_pass_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DETECTION_THRESH = 0.3
PAGES_PER_PDF    = 1
SEED             = 42

WEIGHTS_PATH = Path.home() / "newspaper_navigator_model" / "model_final.pth"
NUM_CLASSES  = 7
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_CLASS_IDS = {0, 1, 2, 3, 4}
LABEL_NAMES     = ["Photograph", "Illustration", "Map",
                   "Comic", "EditorialCartoon", "Headline", "Advertisement"]

CLASS_COL_MAP = {
    0: "photograph",
    1: "illustration",
    2: "map",
    3: "comic",
    4: "editorial_cartoon",
}

print(f"Device : {DEVICE}")
print(f"PDF_DIR : {PDF_DIR}")
print(f"Output : {OUTPUT_DIR}")

Device : cuda
PDF_DIR : /content/drive/Othercomputers/My Laptop/pdfs
Output : /content/drive/MyDrive/detection_pass_results


### Pull All Front Pages

In [29]:
all_pdfs = sorted(PDF_DIR.rglob("*.pdf"))
pages = []
for pdf in all_pdfs:
    pages.append({
        "archive_path": str(pdf.relative_to(PDF_DIR)),
        "year":         pdf.parts[-3],
        "page_num":     0,
        "local_path":   str(pdf),
    })

print(f"Queued {len(pages)} pages from {len(all_pdfs)} PDFs")

Queued 416 pages from 416 PDFs


### Model Setup

In [30]:
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.model_zoo import model_zoo

def build_predictor(score_thresh: float) -> DefaultPredictor:
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.WEIGHTS                     = str(WEIGHTS_PATH)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ROI_HEADS.NUM_CLASSES       = NUM_CLASSES
    cfg.MODEL.DEVICE                      = DEVICE
    cfg.INPUT.MIN_SIZE_TEST               = 800
    cfg.INPUT.MAX_SIZE_TEST               = 1333
    return DefaultPredictor(cfg)

predictor = build_predictor(DETECTION_THRESH)
print(f"Predictor ready (thresh={DETECTION_THRESH}, device={DEVICE})")

Predictor ready (thresh=0.3, device=cuda)


### Run Inference & Save Detections

In [31]:
import json
import numpy as np
from pdf2image import convert_from_path
from tqdm import tqdm

detections = []

for p in tqdm(pages, desc="Inference"):
    pdf_path = PDF_DIR / p["archive_path"]
    pg = p["page_num"]
    try:
        imgs = convert_from_path(str(pdf_path), dpi=150,
                                 first_page=pg + 1, last_page=pg + 1)
        arr = np.array(imgs[0])[:, :, ::-1]   # RGB -> BGR
        out  = predictor(arr)
        inst = out["instances"].to("cpu")
        detections.append({
            "archive_path": p["archive_path"],
            "year":         p["year"],
            "page_num":     pg,
            "img_w":        imgs[0].width,
            "img_h":        imgs[0].height,
            "scores":       inst.scores.numpy().tolist(),
            "classes":      inst.pred_classes.numpy().tolist(),
            "boxes":        inst.pred_boxes.tensor.numpy().tolist(),
        })
    except Exception as e:
        print(f"  [WARN] {p['archive_path']} p{pg}: {e}")
        detections.append({
            "archive_path": p["archive_path"],
            "year":         p["year"],
            "page_num":     pg,
            "img_w":        None,
            "img_h":        None,
            "scores":       [],
            "classes":      [],
            "boxes":        [],
        })

out_path = OUTPUT_DIR / "detections_review.json"
with open(out_path, "w") as f:
    json.dump(detections, f)

print(f"{len(detections)} pages -> detections_review.json")
print("Download this file and place it next to review_labels.py")

Inference:   0%|          | 0/416 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0426 20:03:38.995000 22156 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Inference: 100%|██████████| 416/416 [31:20<00:00,  4.52s/it]


416 pages -> detections_review.json
Download this file and place it next to review_labels.py
